In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json

In [21]:
df = pd.read_csv('../data/df_transaction_clean.csv')
df.head()

,id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,errors,mcc_code,categoria
0,7475327,2010-01-01 00:01:00,1556,2972,-77.00,Swipe Transaction,59935,Beulah,ND,58523.0,5499,Sin Error,5499,Miscellaneous Food Stores
1,7475328,2010-01-01 00:02:00,561,4575,14.57,Swipe Transaction,67570,Bettendorf,IA,52722.0,5311,Sin Error,5311,Department Stores
2,7475329,2010-01-01 00:02:00,1129,102,80.00,Swipe Transaction,27092,Vista,CA,92084.0,4829,Sin Error,4829,Money Transfer
3,7475331,2010-01-01 00:05:00,430,2860,200.00,Swipe Transaction,27092,Crown Point,IN,46307.0,4829,Sin Error,4829,Money Transfer
4,7475332,2010-01-01 00:06:00,848,3915,46.41,Swipe Transaction,13051,Harwood,MD,20776.0,5813,Sin Error,5813,Drinking Places (Alcoholic Beverages)


In [22]:
df_2 = df.copy()

### Pregunta 11 - ¿Existen transacciones marcadas como fraudulentas?¿Qué porcentaje representan?
- El porcentaje de las transacciones marcadas como fraudulentas es del 0.15%

In [3]:
with open('../data/train_fraud_labels.json') as f:
    fraud_labels = json.load(f)

fraud_inner = fraud_labels['target']
df_fraud = pd.DataFrame(list(fraud_inner.items()), columns=['id', 'is_fraud'])
df_fraud.head()

,id,is_fraud
0,10649266,No
1,23410063,No
2,9316588,No
3,12478022,No
4,9558530,No


In [4]:
df_fraud.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8914963 entries, 0 to 8914962
Data columns (total 2 columns):
 #   Column    Dtype 
---  ------    ----- 
 0   id        object
 1   is_fraud  object
dtypes: object(2)
memory usage: 136.0+ MB


In [23]:
df_2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13305915 entries, 0 to 13305914
Data columns (total 14 columns):
 #   Column          Dtype  
---  ------          -----  
 0   id              int64  
 1   date            object 
 2   client_id       int64  
 3   card_id         int64  
 4   amount          float64
 5   use_chip        object 
 6   merchant_id     int64  
 7   merchant_city   object 
 8   merchant_state  object 
 9   zip             object 
 10  mcc             int64  
 11  errors          object 
 12  mcc_code        int64  
 13  categoria       object 
dtypes: float64(1), int64(6), object(7)
memory usage: 1.4+ GB


In [24]:
# Cambiar el tipo de datos del id en int
df_fraud['id'] = df_fraud['id'].astype(int)
# Merge los dos df
df_2 = df_2.merge(df_fraud, on='id', how='left')
df_2.head()

,id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,errors,mcc_code,categoria,is_fraud
0,7475327,2010-01-01 00:01:00,1556,2972,-77.00,Swipe Transaction,59935,Beulah,ND,58523.0,5499,Sin Error,5499,Miscellaneous Food Stores,No
1,7475328,2010-01-01 00:02:00,561,4575,14.57,Swipe Transaction,67570,Bettendorf,IA,52722.0,5311,Sin Error,5311,Department Stores,No
2,7475329,2010-01-01 00:02:00,1129,102,80.00,Swipe Transaction,27092,Vista,CA,92084.0,4829,Sin Error,4829,Money Transfer,No
3,7475331,2010-01-01 00:05:00,430,2860,200.00,Swipe Transaction,27092,Crown Point,IN,46307.0,4829,Sin Error,4829,Money Transfer,NaN
4,7475332,2010-01-01 00:06:00,848,3915,46.41,Swipe Transaction,13051,Harwood,MD,20776.0,5813,Sin Error,5813,Drinking Places (Alcoholic Beverages),No


In [25]:
df_2['is_fraud'].value_counts()

is_fraud
No     8901631
Yes      13332
Name: count, dtype: int64

In [26]:
df_2.isnull().sum()

id                      0
date                    0
client_id               0
card_id                 0
amount                  0
use_chip                0
merchant_id             0
merchant_city           0
merchant_state          0
zip                     0
mcc                     0
errors                  0
mcc_code                0
categoria               0
is_fraud          4390952
dtype: int64

In [28]:
# Manejo de valores nulos en 'is_fraud'
df_2['is_fraud'] = df_2['is_fraud'].map({"Yes": 1, "No": 0}).fillna(-1).astype(int)
df_2.head()

,id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,errors,mcc_code,categoria,is_fraud
0,7475327,2010-01-01 00:01:00,1556,2972,-77.00,Swipe Transaction,59935,Beulah,ND,58523.0,5499,Sin Error,5499,Miscellaneous Food Stores,0
1,7475328,2010-01-01 00:02:00,561,4575,14.57,Swipe Transaction,67570,Bettendorf,IA,52722.0,5311,Sin Error,5311,Department Stores,0
2,7475329,2010-01-01 00:02:00,1129,102,80.00,Swipe Transaction,27092,Vista,CA,92084.0,4829,Sin Error,4829,Money Transfer,0
3,7475331,2010-01-01 00:05:00,430,2860,200.00,Swipe Transaction,27092,Crown Point,IN,46307.0,4829,Sin Error,4829,Money Transfer,-1
4,7475332,2010-01-01 00:06:00,848,3915,46.41,Swipe Transaction,13051,Harwood,MD,20776.0,5813,Sin Error,5813,Drinking Places (Alcoholic Beverages),0


In [29]:
fraud_transaction = df_2[df_2['is_fraud'] != -1]

total_fraud_transaction = len(fraud_transaction)
total_fraud = fraud_transaction['is_fraud'].sum()
# Calcular el porcentaje de las transacciones marcadas como fraudulentas
porcentaje_fraud = (total_fraud / total_fraud_transaction) * 100
print(f"Porcentaje de fraude: {porcentaje_fraud:.2f}%")

Porcentaje de fraude: 0.15%


### Pregunta 12 - ¿Las transacciones fraudulentas tienen montos más altos que las normales en promedio?
- Promedio del monto Transacciones Legitima
    - $45.85

- Promedio monto transacciones Fraudulentas
    - $110.23 

In [10]:
monto_fraude = (fraud_transaction
    .groupby('is_fraud')['amount']
    .mean()
    .round(2)
    .reset_index()
)
monto_fraude['tipo'] = monto_fraude['is_fraud'].map({0: 'Legitima', 1: 'Fraudulenta'})
monto_fraude.head()

,is_fraud,amount,tipo
0,0,42.85,Legitima
1,1,110.23,Fraudulenta


 ### Pregunta 13 - ¿Hay alguna categoría con una tasa de fraude desproporcionadamente alta?
 - Cruise Lines con un 59.78%

In [11]:
fraude_cat = (fraud_transaction
    .groupby('categoria')
    .agg(
        total_transaction=('is_fraud', 'count'),
        total_fraud=('is_fraud', 'sum')
        )
    .reset_index()
)
fraude_cat['porcentaje'] = (fraude_cat['total_fraud']/fraude_cat['total_transaction']*100).round(2)
fraude_cat.head()

,categoria,total_transaction,total_fraud,porcentaje
0,"Accounting, Auditing, and Bookkeeping Services",2354,0,0.00
1,Airlines,1913,38,1.99
2,"Amusement Parks, Carnivals, Circuses",7735,123,1.59
3,Antique Shops,3977,86,2.16
4,"Artist Supply Stores, Craft Shops",45281,78,0.17


In [12]:
fraude_cat = fraude_cat.sort_values(by='porcentaje', ascending=False)
fraude_cat.head()

,categoria,total_transaction,total_fraud,porcentaje
24,Cruise Lines,276,165,59.78
73,Music Stores - Musical Instruments,204,76,37.25
63,Miscellaneous Fabricated Metal Products,245,29,11.84
22,"Computers, Computer Peripheral Equipment",1883,204,10.83
40,Floor Covering Stores,222,23,10.36


### Pregunta 14 - ¿Existen transacciones con montos atípicos (outliers)? Usa el método IQR para identificarlos.
- IRQ:  55.33
- Estan son las categorias con el valor mas alto de los montos atipicos
    - Service Stations: 177463
    - Miscellaneous Food Stores: 166692
- Tasa de fraude en outliers: 0.54%
- Tasa de fraude general: 0.15%

In [31]:
df_outliers = df_2[df_2['amount'] > 0]['amount']
# Definir IRQ
q1 = df_outliers.quantile(0.25)
q3 = df_outliers.quantile(0.75)
IRQ = q3 - q1
limite_inferior = q1 - 1.5 * IRQ
limite_superior = q3 + 1.5 * IRQ

# Resultados
print("Q1: ", q1)
print("Q3: ", q3)
print("IRQ: ", round(IRQ, 2))
print("Limite Inferior: ", round(limite_inferior, 2))
print("Limite Superior: ", round(limite_superior, 2))

Q1:  11.07
Q3:  66.4
IRQ:  55.33
Limite Inferior:  -71.93
Limite Superior:  149.4


In [33]:
outliers = df_2[
      (df_2['amount'] > limite_superior) | 
      (df_2['amount'] < limite_inferior)
      ]
outliers_clasificados = outliers[outliers['is_fraud'] != -1]
tasa_fraude_outliers = (outliers_clasificados['is_fraud'].mean()*100).round(2)
tasa_fraude_general = ((total_fraud / total_fraud_transaction)*100).round(2)

In [35]:
total_outliers = len(outliers)
total_transaccion = len(df_2)
promedio_outliers = round((total_outliers / total_transaccion)*100, 2)
min_outlier = outliers['amount'].min()
max_outlier = outliers['amount'].max()

print("Total Outliers: ", total_outliers)
print("Porcentaje de outliers: ", promedio_outliers,"%")
print("Monto Minimo: ", min_outlier)
print("Monta Maximo: ", max_outlier)

Total Outliers:  1044249
Porcentaje de outliers:  7.85 %
Monto Minimo:  -500.0
Monta Maximo:  6820.2


In [36]:
# Respuestas
print("Categorias donde se enceuntran los outliers: ", outliers['categoria'].value_counts().head())
print('\n')
print(f"Tasa de fraude en outliers: {tasa_fraude_outliers}%")
print(f"Tasa de fraude general: {tasa_fraude_general}%")

Categorias donde se enceuntran los outliers:  categoria
Service Stations                              177463
Miscellaneous Food Stores                     166692
Utilities - Electric, Gas, Water, Sanitary     63432
Telecommunication Services                     59362
Money Transfer                                 41842
Name: count, dtype: int64


Tasa de fraude en outliers: 0.54%
Tasa de fraude general: 0.15%


#### Donde se encuentra los outliers, ¿En los montos altos o en los montos bajos?
- Tasa de fraude outliers, Montos Altos: 0.78%
- Tasa de fraude outliers, Montos Bajos: 0.17%

In [38]:
# Separar outliers superiores e inferiores
outliers_altos = outliers[outliers['amount'] > limite_superior]
outliers_bajos = outliers[outliers['amount'] < limite_inferior]

print(f"Outliers por monto alto: {len(outliers_altos):,}")
print(f"Outliers por monto bajo (devoluciones): {len(outliers_bajos):,}")

Outliers por monto alto: 634,806
Outliers por monto bajo (devoluciones): 409,443


In [39]:
# Tasa de fraude por tipo de outlier
for nombre, df in [('Altos', outliers_altos), ('Bajos', outliers_bajos)]:
    clasificados = df[df['is_fraud'] != -1]
    tasa = clasificados['is_fraud'].mean() * 100
    print(f"Tasa de fraude outliers {nombre}: {tasa:.2f}%")

Tasa de fraude outliers Altos: 0.78%
Tasa de fraude outliers Bajos: 0.17%


### Pregunta 15 - ¿Hay clientes que realizan muchas transacciones pequeñas en poco tiempo? (posible evasión — "smurfing")
- Hay solo dos transacciones de un cliente que es sospechoso
    - Client_id:
        -  1428  
- Transacciones Sospechosas
    - Cliente: 1428 | Fecha: 2016-06-26
    - Transacciones ese día: 11
    - Monto promedio: $5.58 | Std: $4.04
      | id | date | amount | categoria | merchant_id | is_fraud |
      | --- | --- | --- | --- | --- | --- |
      | 11031097 | 2012-04-23 14:55:00 | -98.0 | Miscellaneous Food Stores | 59935 | 0
      | 11031334 | 2012-04-23 15:44:00 | -100.0 | Miscellaneous Food Stores | 59935 | -1
      | 11031899 | 2012-04-23 18:15:00 | -99.0 | Miscellaneous Food Stores | 59935 | 0
      | 17995663 | 2016-06-26 06:11:00 | 1.44 | Miscellaneous Food Stores | 14528 | 0 |
      | 17996439 | 2016-06-26 08:39:00 | 3.63 | Grocery Stores, Supermarkets | 50783 | 0 |
      | 17996542 | 2016-06-26 08:55:00 | 6.55 | Fast Food Restaurants | 44919 | 0 |
      | 17997261 | 2016-06-26 11:10:00 | 11.97 | Postal Services - Government Only | 83480 | -1 |
      | 17998854 | 2016-06-26 15:57:00 | 8.26 | Taxicabs and Limousines | 18563 | 0 |
      | 17998937 | 2016-06-26 16:15:00 | 1.26 | Miscellaneous Food Stores | 14528 | 0 |
      | 17999176 | 2016-06-26 17:05:00 | 1.86 | Miscellaneous Food Stores | 14528 | 0 |
      | 17999350 | 2016-06-26 18:03:00 | 8.14 | Taxicabs and Limousines | 80263 | -1 |
      | 17999425 | 2016-06-26 18:33:00 | 11.32 | Taxicabs and Limousines | 16798 | 0 |
      | 17999437 | 2016-06-26 18:35:00 | 6.03 | Taxicabs and Limousines | 16798 | 0 |
      | 17999576 | 2016-06-26 19:29:00 | 0.95 | Miscellaneous Food Stores | 14528 | 0 |

    - Cliente: 1428 | Fecha: 2019-10-09
    - Transacciones ese día: 11
    - Monto promedio: $7.79 | Std: $4.84
      | id | date | amount | categoria | merchant_id | is_fraud |
      | --- | --- | --- | --- | --- | --- |
      | 23654183 | 2019-10-09 04:17:00 | 18.47 | Department Stores | 9932 | 0 |
      | 23654320 | 2019-10-09 06:10:00 | 1.51 | Miscellaneous Food Stores | 14528 | 0 |
      | 23655720 | 2019-10-09 11:33:00 | 10.57 | Postal Services - Government Only | 83480 | -1 |
      | 23656911 | 2019-10-09 15:29:00 | 8.11 | Taxicabs and Limousines | 16798 | -1 |
      | 23656937 | 2019-10-09 15:34:00 | 8.36 | Taxicabs and Limousines | 16798 | 0 |
      | 23657045 | 2019-10-09 15:55:00 | 2.93 | Medical Services | 70544 | 0 |
      | 23657092 | 2019-10-09 16:06:00 | 8.79 | Taxicabs and Limousines | 80263 | -1 |
      | 23657551 | 2019-10-09 18:00:00 | 6.81 | Taxicabs and Limousines | 16798 | 0 |
      | 23657706 | 2019-10-09 18:55:00 | 9.21 | Taxicabs and Limousines | 18563 | -1 |
      | 23657804 | 2019-10-09 19:21:00 | 1.34 | Miscellaneous Food Stores | 14528 | -1 |
      | 23657856 | 2019-10-09 19:39:00 | 9.62 | Taxicabs and Limousines | 80263 | -1 |

- Los 2 casos identificados con criterios estrictos corresponden al cliente 1428, quien en dos fechas distintas (2016-06-26 y 2019-10-09) realizó 11 transacciones diarias en categorías de gasto cotidiano: alimentación y transporte. Tras análisis manual, se descarta smurfing ya que los patrones de gasto son coherentes con actividad normal urbana — múltiples viajes en taxi y compras de comida a lo largo del día. El algoritmo generó falsos positivos debido a que el umbral de desviación estándar ($5) no discrimina suficientemente entre montos repetitivos intencionalmente y variaciones naturales de gasto cotidiano.


In [42]:
promedio = df_2['amount'].mean()
mediana = df_2['amount'].median()
print(f"Promedio: {promedio:.2f}\nMediana: {mediana:.2f}")

Promedio: 42.98
Mediana: 28.99


In [44]:
df_2['date'] = pd.to_datetime(df_2['date'])
df_2['fecha'] = df_2['date'].dt.date

transacciones_por_dia_cliente = (df_2
    .groupby(['client_id', 'fecha'])
    .size()
)
print(f"Promedio transacciones por cliente por día: {transacciones_por_dia_cliente.mean():.2f}")
print(f"Percentil 95: {transacciones_por_dia_cliente.quantile(0.95):.0f} Transacciones por día")

Promedio transacciones por cliente por día: 3.43
Percentil 95: 8 Transacciones por día


In [46]:
# Agrupar clientes y fechas
transacciones_diarias = (df_2
    .groupby(['client_id','fecha'])
    .agg(
        num_transaction = ('id','count'),
        monto_promedio = ('amount', 'mean'),
        monto_total = ('amount', 'sum')
    )
    .reset_index()
)
limite_transacciones_diarias_cliente = transacciones_diarias['num_transaction'].quantile(0.95)

# Filtrar por cliente
sospechosos = transacciones_diarias[
    (transacciones_diarias['num_transaction'] >= limite_transacciones_diarias_cliente) |
    (transacciones_diarias['monto_total'] <= mediana)
    ].sort_values(by='num_transaction', ascending=False)

In [48]:
sospechosos.head()

,client_id,fecha,num_transaction,monto_promedio,monto_total
2082181,1098,2013-11-15,31,42.068710,1304.13
2083553,1098,2017-08-19,31,33.563548,1040.47
230299,114,2011-12-29,29,15.428621,447.43
3285430,1696,2011-06-12,29,31.415862,911.06
3653130,1888,2010-12-22,28,15.279286,427.82


In [50]:
print(f"Umbral transacciones: {limite_transacciones_diarias_cliente:.0f} por día")
print(f"Umbral monto:         ${mediana}")
print(f"\nRegistros sospechosos encontrados: {len(sospechosos):,}")

Umbral transacciones: 8 por día
Umbral monto:         $28.99

Registros sospechosos encontrados: 897,927


In [53]:
# Verificar si hay fraude en los sospechosos
cliente_sospechoso = sospechosos['client_id'].unique()

df_sospechoso = df_2[
    (df_2['client_id'].isin(cliente_sospechoso)) & 
    (df_2['is_fraud'] != -1)
    ]
tasa_fraude_promedio = df_sospechoso['is_fraud'].mean() * 100

print(f"Tasa de fraude en clientes sospechosos: {tasa_fraude_promedio:.2f}%")
print(f"Tasa de fraude general:{tasa_fraude_general}%")

Tasa de fraude en clientes sospechosos: 0.15%
Tasa de fraude general:0.15%


In [56]:
# Buscar con umbral mas estricto
# En lugar del percentil 95, usa el 99
limite_transacciones_99 = transacciones_por_dia_cliente.quantile(0.99)

# En lugar de la mediana, usa un monto más bajo (percentil 25)
monto_muy_pequeno = df_2['amount'].quantile(0.25)

print(f"Nuevo umbral transacciones (p99): {limite_transacciones_99:.0f} por día")
print(f"Nuevo umbral monto (p25):         ${monto_muy_pequeno:.2f}")

# Recalcular sospechosos
sospechosos_estricto = transacciones_diarias[
    (transacciones_diarias['num_transaction'] >= limite_transacciones_99) &
    (transacciones_diarias['monto_promedio'] <= monto_muy_pequeno)
].sort_values('num_transaction', ascending=False)

print(f"\nRegistros sospechosos (criterio estricto): {len(sospechosos_estricto):,}")

Nuevo umbral transacciones (p99): 11 por día
Nuevo umbral monto (p25):         $8.93

Registros sospechosos (criterio estricto): 1,361


In [60]:
patron_smurfing_v2 = (df_2
    .groupby(['client_id', 'fecha'])
    .agg(
        num_transacciones=('id', 'count'),
        monto_promedio=('amount', 'mean'),
        monto_std=('amount', 'std'),      # ← nuevo criterio
        monto_total=('amount', 'sum')
    )
    .reset_index()
)

# Sospechoso = muchas transacciones + montos bajos + montos muy similares
sospechosos_v2 = patron_smurfing_v2[
    (patron_smurfing_v2['num_transacciones'] >= limite_transacciones_99) &
    (patron_smurfing_v2['monto_promedio'] <= monto_muy_pequeno) &
    (patron_smurfing_v2['monto_std'] <= 5)   # montos casi idénticos
].sort_values('num_transacciones', ascending=False)

print(f"Sospechosos con criterio de repetición: {len(sospechosos_v2):,}")

Sospechosos con criterio de repetición: 2


In [63]:
# Analisis de esos 2
# Ver quiénes son esos 2 clientes
print(sospechosos_v2[['client_id', 'fecha', 'num_transacciones', 'monto_promedio', 'monto_std', 'monto_total']])

# Ver todas sus transacciones de ese día específico
for _, row in sospechosos_v2.iterrows():
    transacciones_dia = df_2[
        (df_2['client_id'] == row['client_id']) &   
        (df_2['fecha'] == row['fecha'])][['id', 'date', 'amount', 'categoria', 'merchant_id', 'is_fraud']]
    
    print(f"\nCliente: {row['client_id']} | Fecha: {row['fecha']}")
    print(f"Transacciones ese día: {row['num_transacciones']}")
    print(f"Monto promedio: ${row['monto_promedio']:.2f} | Std: ${row['monto_std']:.2f}")
    print(transacciones_dia.to_string())

         client_id       fecha  num_transacciones  monto_promedio  monto_std  \
2721423       1428  2016-06-26                 11        5.582727   4.043993   
2722614       1428  2019-10-09                 11        7.792727   4.839465   

         monto_total  
2721423        61.41  
2722614        85.72  

Cliente: 1428 | Fecha: 2016-06-26
Transacciones ese día: 11
Monto promedio: $5.58 | Std: $4.04
               id                date  amount                          categoria  merchant_id  is_fraud
8629892  17995663 2016-06-26 06:11:00    1.44          Miscellaneous Food Stores        14528         0
8630522  17996439 2016-06-26 08:39:00    3.63       Grocery Stores, Supermarkets        50783         0
8630599  17996542 2016-06-26 08:55:00    6.55              Fast Food Restaurants        44919         0
8631178  17997261 2016-06-26 11:10:00   11.97  Postal Services - Government Only        83480        -1
8632472  17998854 2016-06-26 15:57:00    8.26            Taxicabs and Limo

In [ ]:
df_2.to_csv(r"C:\Users\Personal\Documents\PythonDataAnalisis\Portafolio\Python_Projects\Analisis Financiero\data\df_clean_graficos.csv", index=False)
fraude_cat.to_csv(r"C:\Users\Personal\Documents\PythonDataAnalisis\Portafolio\Python_Projects\Analisis Financiero\data\df_categoria_fraude.csv", index=False)